# RSNA Knee Abnormality Detection — Experiment Notebook

**Purpose:** keep the reproducible experiments in one ordered notebook and remove exploratory/debugging clutter.

## How to use this notebook

- **GPU OFF:** dataset-label audit, report inspection, logistic-regression experiments, and weak-label generation/audit.
- **GPU ON:** DINOv2 feature extraction (EXP-004) and test-study feature extraction for the final submission (EXP-008). Both explicitly require CUDA.
- Do **not** use **Run All** by default. Run the dataset audit first. Run feature extraction only if the cache is missing; otherwise skip it and validate the cache.
- The cache and weak-label outputs live under `/kaggle/working/` and are temporary runtime files. Download important outputs before ending the session.
- The fixed gold-label split is `SPLIT-42-ACL-43-15`: 43 train studies / 15 validation studies, seed 42, stratified by ACL.
- Only 58 studies have all 12 official labels. The other 4,349 studies have all targets missing; missing labels are **not negatives**.
- Validation metrics are highly uncertain because the labelled validation set has only 15 studies. Treat small differences as exploratory, not conclusive.

## 1. Dataset and label audit

Run with **GPU OFF**. This verifies the central label-missingness fact before experiments.

In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
df = pd.read_csv(DATA / "train.csv")

targets = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("Studies:", len(df))
print("\nObserved labels per target:")
print(df[targets].notna().sum().to_string())

print("\nMissing labels per target:")
print(df[targets].isna().sum().to_string())

observed_per_study = df[targets].notna().sum(axis=1)
print("\nObserved labels per study:")
print(observed_per_study.value_counts().sort_index().to_string())

print("\nStudies with at least one observed target:",
      int((observed_per_study > 0).sum()))
print("Studies with all 12 targets observed:",
      int((observed_per_study == 12).sum()))

print("\nLabel value counts, including missing values:")
for target in targets:
    print(f"\n{target}")
    print(df[target].value_counts(dropna=False).to_string())


## 2. Historical baseline results

These are recorded results from the earlier notebook. This section is a record only; it does not rerun the old CNNs.

| Experiment | Method | Validation macro-AUC |
|---|---|---:|
| EXP-001 | Three-slice CNN, slice predictions averaged per study | 0.5018 |
| EXP-002 | Study-level CNN, 3 slices as channels | 0.4628 |
| EXP-003 | Study-level CNN, 9 slices as channels | 0.5684 |
| EXP-004 | Frozen DINOv2 Base embeddings + logistic regression | 0.5717 |
| EXP-005 | Report text TF-IDF + logistic regression | 0.5351 |
| EXP-006 | DINOv2 embeddings + report TF-IDF | 0.5665 |

**Interpretation:** EXP-004 is the highest of these recorded baselines, but the validation sample is too small to conclude that the difference is robust. Do not compare these local results directly with leaderboard scores without checking evaluation protocols.

## 3. EXP-004 — DINOv2 feature extraction

**GPU ON.** This cell extracts features for the 58 fully labelled studies and writes a cache to `/kaggle/working/exp004_dinov2_cache`. Skip this cell if that cache already exists and is complete.

In [1]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import pydicom
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

# ---------------- Configuration ----------------
DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
MODEL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/base/1"
OUT = Path("/kaggle/working/exp004_dinov2_cache")
OUT.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
N_SLICES = 9
SEED = 42

# ---------------- Data and split ----------------
labels = pd.read_csv(DATA / "train.csv")
series_df = pd.read_csv(DATA / "train_series.csv")

labelled = labels.dropna(subset=TARGETS).copy()
labelled = labelled.sort_values("StudyInstanceUID").reset_index(drop=True)

from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    labelled["StudyInstanceUID"].tolist(),
    test_size=0.25,
    random_state=SEED,
    stratify=labelled["ACL"]
)
split = {study: "train" for study in train_ids}
split.update({study: "validation" for study in val_ids})

assert len(labelled) == 58
assert len(train_ids) == 43 and len(val_ids) == 15

# ---------------- Model ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("GPU is not enabled. Turn the Kaggle accelerator on.")

processor = AutoImageProcessor.from_pretrained(
    MODEL_PATH, local_files_only=True
)
model = AutoModel.from_pretrained(
    MODEL_PATH, local_files_only=True
).to(device).eval()

print("Device:", device)
print("Studies:", len(labelled), "| train:", len(train_ids), "| validation:", len(val_ids))

# ---------------- Series selection ----------------
def choose_series(study_id):
    candidates = series_df[series_df["StudyInstanceUID"] == study_id]
    sagittal = candidates[
        (candidates["Anatomical_Plane"] == "Sagittal")
        & (candidates["Fluid_Sensitive"] == 1)
    ]
    coronal = candidates[
        (candidates["Anatomical_Plane"] == "Coronal")
        & (candidates["Fluid_Sensitive"] == 1)
    ]

    if not sagittal.empty:
        return sagittal.sort_values("SeriesInstanceUID", kind="stable").iloc[0], "sagittal_fluid"
    if not coronal.empty:
        return coronal.sort_values("SeriesInstanceUID", kind="stable").iloc[0], "coronal_fluid_fallback"

    raise RuntimeError(f"No eligible fluid-sensitive series for {study_id}")

def load_ordered_dicom_paths(study_id, series_id):
    folder = DATA / "train_series" / str(study_id) / str(series_id)
    paths = list(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM files in {folder}")

    # Read metadata and order slices deterministically.
    entries = []
    for path in paths:
        ds = pydicom.dcmread(path, stop_before_pixels=True)
        instance = getattr(ds, "InstanceNumber", None)
        if instance is None:
            raise ValueError(f"Missing InstanceNumber: {path}")
        entries.append((int(instance), path))

    entries.sort(key=lambda x: (x[0], x[1].name))
    instances = [x[0] for x in entries]
    if len(instances) != len(set(instances)):
        raise ValueError(f"Duplicate InstanceNumber in {study_id}/{series_id}")

    return [x[1] for x in entries]

def dicom_to_image(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1))
    pixels += float(getattr(ds, "RescaleIntercept", 0))

    lo, hi = np.percentile(pixels, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        raise ValueError(f"Invalid intensity range: {path}")

    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
    pixels = (pixels * 255).astype(np.uint8)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = 255 - pixels

    return Image.fromarray(pixels).convert("RGB"), ds

# ---------------- Extract features ----------------
all_features = []
metadata = []
start = time.time()

for study_idx, row in labelled.iterrows():
    study_id = row["StudyInstanceUID"]
    chosen, selection = choose_series(study_id)
    series_id = chosen["SeriesInstanceUID"]

    paths = load_ordered_dicom_paths(study_id, series_id)
    # Nine deterministic, evenly spaced indices, including both ends.
    indices = np.rint(np.linspace(0, len(paths) - 1, N_SLICES)).astype(int)
    if len(set(indices.tolist())) != N_SLICES:
        raise RuntimeError(f"Repeated slice indices for {study_id}")

    images = []
    slice_metadata = []

    for idx in indices:
        path = paths[int(idx)]
        image, ds = dicom_to_image(path)
        images.append(image)
        slice_metadata.append({
            "study_id": study_id,
            "series_id": series_id,
            "selection": selection,
            "slice_index_in_sorted_series": int(idx),
            "instance_number": int(ds.InstanceNumber),
            "dicom_filename": path.name,
            "split": split[study_id],
        })

    # Process one study (9 slices) at a time to limit GPU memory.
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model(**inputs)
        # CLS token: one 768-dimensional vector per slice.
        feats = outputs.last_hidden_state[:, 0, :]

    feats = feats.float().cpu().numpy()
    if feats.shape != (N_SLICES, 768) or not np.isfinite(feats).all():
        raise RuntimeError(f"Unexpected/invalid features for {study_id}: {feats.shape}")

    all_features.append(feats)
    metadata.extend(slice_metadata)

    if (study_idx + 1) % 5 == 0 or study_idx + 1 == len(labelled):
        print(
            f"{study_idx + 1}/{len(labelled)} studies | "
            f"{time.time() - start:.1f}s elapsed"
        )

# Shape: studies x slices x embedding dimensions
features = np.stack(all_features).astype(np.float32)
study_ids = labelled["StudyInstanceUID"].tolist()

np.save(OUT / "features.npy", features)
np.save(OUT / "study_ids.npy", np.asarray(study_ids))
pd.DataFrame(metadata).to_csv(OUT / "slice_metadata.csv", index=False)

# Labels and split manifest are saved separately for traceability.
labelled[["StudyInstanceUID"] + TARGETS].to_csv(OUT / "labels.csv", index=False)
pd.DataFrame({
    "StudyInstanceUID": study_ids,
    "split": [split[s] for s in study_ids],
}).to_csv(OUT / "split_manifest.csv", index=False)

manifest = {
    "experiment": "EXP-004",
    "model": "DINOv2 Base",
    "model_path": MODEL_PATH,
    "n_studies": len(study_ids),
    "n_slices_per_study": N_SLICES,
    "feature_dim": int(features.shape[-1]),
    "feature_shape": list(features.shape),
    "split_seed": SEED,
    "train_studies": len(train_ids),
    "validation_studies": len(val_ids),
    "selection_policy": "fluid-sensitive sagittal; coronal fallback",
    "slice_sampling": "9 rounded evenly spaced indices after InstanceNumber ordering",
    "preprocessing": "per-slice 1st/99th percentile scaling, MONOCHROME1 inversion, RGB",
    "feature": "DINOv2 last_hidden_state CLS token",
    "elapsed_seconds": round(time.time() - start, 2),
}

with open(OUT / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nExtraction complete.")
print("Feature shape:", features.shape)
print("Feature cache:", OUT)
print("Finite values:", bool(np.isfinite(features).all()))
print("Elapsed seconds:", manifest["elapsed_seconds"])


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Device: cuda
Studies: 58 | train: 43 | validation: 15
5/58 studies | 3.1s elapsed
10/58 studies | 5.5s elapsed
15/58 studies | 7.9s elapsed
20/58 studies | 10.9s elapsed
25/58 studies | 12.8s elapsed
30/58 studies | 14.7s elapsed
35/58 studies | 17.0s elapsed
40/58 studies | 21.8s elapsed
45/58 studies | 24.1s elapsed
50/58 studies | 26.2s elapsed
55/58 studies | 28.4s elapsed
58/58 studies | 29.7s elapsed

Extraction complete.
Feature shape: (58, 9, 768)
Feature cache: /kaggle/working/exp004_dinov2_cache
Finite values: True
Elapsed seconds: 29.74


In [1]:
from pathlib import Path
import zipfile

p = Path("/kaggle/working/exp004_dinov2_cache_backup.zip")
print("Exists:", p.exists())
print("Size:", p.stat().st_size if p.exists() else "N/A")
print("Valid ZIP:", zipfile.is_zipfile(p))

Exists: True
Size: 1512987
Valid ZIP: True


## 4. Verify the DINOv2 cache

**GPU OFF.** Run this before any cached-feature experiment. If the cache is absent, return to EXP-004 and run feature extraction with GPU ON.

In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

CACHE = Path("/kaggle/working/exp004_dinov2_cache")

features = np.load(CACHE / "features.npy")
study_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True)
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")
slice_metadata = pd.read_csv(CACHE / "slice_metadata.csv")
manifest = json.loads((CACHE / "manifest.json").read_text())

print("Features:", features.shape)
print("Studies:", len(study_ids))
print("Unique study IDs:", len(set(study_ids)))
print("Labels:", labels.shape)
print("Split rows:", len(split))
print("Slice metadata rows:", len(slice_metadata))
print("Finite features:", np.isfinite(features).all())
print("Train studies:", (split["split"] == "train").sum())
print("Validation studies:", (split["split"] == "validation").sum())
print("Cache files:", sorted(p.name for p in CACHE.iterdir()))
print("Manifest:", manifest)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/exp004_dinov2_cache/features.npy'

## 5. EXP-004 — Image-only classifier

**GPU OFF.** Fits logistic regression on cached DINOv2 study embeddings using the fixed split.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from pathlib import Path

CACHE = Path("/kaggle/working/exp004_dinov2_cache")
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

features = np.load(CACHE / "features.npy")
study_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True).astype(str)
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")

# One feature vector per study: average the nine slice embeddings.
study_features = features.mean(axis=1)
feature_df = pd.DataFrame(study_features)
feature_df.insert(0, "StudyInstanceUID", study_ids)

# Merge by study ID rather than relying on row order.
data = (
    split.merge(labels, on="StudyInstanceUID", validate="one_to_one")
         .merge(feature_df, on="StudyInstanceUID", validate="one_to_one")
)

feature_cols = list(range(768))
train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("validation").to_numpy()

assert train_mask.sum() == 43 and val_mask.sum() == 15
assert data["StudyInstanceUID"].is_unique

results = []

for target in TARGETS:
    # Missing labels are excluded, never converted to zero.
    y = pd.to_numeric(data[target], errors="coerce").to_numpy()
    known = np.isfinite(y)
    train_idx = train_mask & known
    val_idx = val_mask & known

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    # Logistic regression requires both classes in training;
    # AUC requires both classes in validation.
    if len(y_train) == 0 or len(np.unique(y_train)) < 2:
        print(f"{target}: SKIPPED — training labels lack both classes")
        continue

    if len(y_val) == 0 or len(np.unique(y_val)) < 2:
        print(f"{target}: AUC unavailable — validation labels lack both classes")
        continue

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )

    model.fit(data.loc[train_idx, feature_cols], y_train)
    probabilities = model.predict_proba(
        data.loc[val_idx, feature_cols]
    )[:, 1]

    auc = roc_auc_score(y_val, probabilities)

    results.append({
        "target": target,
        "train_known": int(train_idx.sum()),
        "validation_known": int(val_idx.sum()),
        "validation_positive": int(y_val.sum()),
        "validation_negative": int((y_val == 0).sum()),
        "auc": float(auc)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

if not results_df.empty:
    macro_auc = results_df["auc"].mean()
    print(f"\nEXP-004 macro-AUC across evaluable targets: {macro_auc:.4f}")
    print("EXP-003 reference macro-AUC: 0.5684")
    print(f"Difference: {macro_auc - 0.5684:+.4f}")

results_df.to_csv(CACHE / "exp004_logistic_results.csv", index=False)
print(f"\nResults saved to {CACHE / 'exp004_logistic_results.csv'}")


## 6. Inspect reports

**GPU OFF.** Displays a small sample of reports to support careful development of report-derived labels. This is an inspection step, not model training.

In [1]:

from pathlib import Path
import pandas as pd

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
train = pd.read_csv(DATA / "train.csv")

print("Dataset shape:", train.shape)
print("Columns:", train.columns.tolist())
print("\nMissing reports:", train["Report"].isna().sum())
print("Empty reports:", train["Report"].fillna("").str.strip().eq("").sum())

# Inspect a few reports without changing the data.
for i, row in train[["StudyInstanceUID", "Report"]].head(5).iterrows():
    report = "" if pd.isna(row["Report"]) else str(row["Report"])
    print(f"\n{'=' * 70}")
    print("Study:", row["StudyInstanceUID"])
    print("Report length:", len(report))
    print(report[:2500])


Dataset shape: (4407, 14)
Columns: ['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

Missing reports: 0
Empty reports: 0

Study: 1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260
Report length: 322
Técnica: RMN de la rodilla. Resultados: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medial. Artrosis femorotibial medial. Derrame. . Impresión: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medial. Artrosis femorotibial medial. Derrame.

Study: 1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585
Report length: 245
[DATE]: * MR Knie Rechts 15ch AA Klinische Inlichtingen: [DATE]. Diagnostische vraagstellling: Meniscusscheur/mediaal? Scanprotocol (DRB) : sag intermediair gewogen seq zonder en met fs, ax/ cor pd gewogen seq fs, cor T1 gewogen seq Bevindi

## 7. EXP-005 — Report-text baseline

**GPU OFF.** Fits TF-IDF on training reports only, then evaluates on the same fixed validation split.

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
CACHE = Path("/kaggle/working/exp004_dinov2_cache")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Reuse the exact study split from EXP-004.
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")
train = pd.read_csv(DATA / "train.csv")

data = (
    split.merge(train[["StudyInstanceUID", "Report"]], on="StudyInstanceUID",
                validate="one_to_one")
         .merge(labels, on="StudyInstanceUID", validate="one_to_one")
)

train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("validation").to_numpy()

assert train_mask.sum() == 43 and val_mask.sum() == 15
assert data["StudyInstanceUID"].is_unique
assert data["Report"].notna().all()

# Fit TF-IDF on training reports only to prevent vocabulary leakage.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=20000,
    sublinear_tf=True
)

X_train_all = vectorizer.fit_transform(data.loc[train_mask, "Report"])
X_val_all = vectorizer.transform(data.loc[val_mask, "Report"])

results = []

for target in TARGETS:
    y = pd.to_numeric(data[target], errors="coerce").to_numpy()
    known = np.isfinite(y)
    train_idx = train_mask & known
    val_idx = val_mask & known

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    if len(np.unique(y_train)) < 2:
        print(f"{target}: SKIPPED — training labels lack both classes")
        continue

    if len(y_val) == 0 or len(np.unique(y_val)) < 2:
        print(f"{target}: AUC unavailable — validation lacks both classes")
        continue

    # Select the matching rows from the train-only and validation matrices.
    train_positions = np.flatnonzero(train_idx[train_mask])
    val_positions = np.flatnonzero(val_idx[val_mask])

    model = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
    model.fit(X_train_all[train_positions], y_train)
    probabilities = model.predict_proba(X_val_all[val_positions])[:, 1]
    auc = roc_auc_score(y_val, probabilities)

    results.append({
        "target": target,
        "train_known": int(train_idx.sum()),
        "validation_known": int(val_idx.sum()),
        "validation_positive": int(y_val.sum()),
        "validation_negative": int((y_val == 0).sum()),
        "auc": float(auc)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

if not results_df.empty:
    macro_auc = results_df["auc"].mean()
    print(f"\nEXP-005 text macro-AUC: {macro_auc:.4f}")
    print("EXP-003 image baseline: 0.5684")
    print("EXP-004 DINOv2 image baseline: 0.5717")
    print(f"Difference vs EXP-004: {macro_auc - 0.5717:+.4f}")

results_df.to_csv(CACHE / "exp005_text_results.csv", index=False)
print(f"\nSaved results to {CACHE / 'exp005_text_results.csv'}")


## 8. EXP-006 — Image + report-text baseline

**GPU OFF.** Combines the image embeddings and report text. Treat this as exploratory given the very small labelled validation set.

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
CACHE = Path("/kaggle/working/exp004_dinov2_cache")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Load cached image embeddings and fixed split.
features = np.load(CACHE / "features.npy").mean(axis=1)
study_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True).astype(str)
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")
reports = pd.read_csv(DATA / "train.csv")[["StudyInstanceUID", "Report"]]

image_df = pd.DataFrame(features, columns=[f"img_{i}" for i in range(768)])
image_df.insert(0, "StudyInstanceUID", study_ids)

data = (
    split.merge(labels, on="StudyInstanceUID", validate="one_to_one")
         .merge(reports, on="StudyInstanceUID", validate="one_to_one")
         .merge(image_df, on="StudyInstanceUID", validate="one_to_one")
)

train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("validation").to_numpy()

assert train_mask.sum() == 43 and val_mask.sum() == 15
assert data["StudyInstanceUID"].is_unique

# Fit text vocabulary using training reports only.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=20000,
    sublinear_tf=True
)

X_text_train = vectorizer.fit_transform(data.loc[train_mask, "Report"])
X_text_val = vectorizer.transform(data.loc[val_mask, "Report"])

# Standardise image features using training studies only.
scaler = StandardScaler()
X_img_train = scaler.fit_transform(
    data.loc[train_mask, [f"img_{i}" for i in range(768)]]
)
X_img_val = scaler.transform(
    data.loc[val_mask, [f"img_{i}" for i in range(768)]]
)

# Combine the two modalities.
from scipy.sparse import csr_matrix, hstack

X_train = hstack([csr_matrix(X_img_train), X_text_train]).tocsr()
X_val = hstack([csr_matrix(X_img_val), X_text_val]).tocsr()

results = []

for target in TARGETS:
    y = pd.to_numeric(data[target], errors="coerce").to_numpy()
    known = np.isfinite(y)
    train_idx = train_mask & known
    val_idx = val_mask & known

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    if len(np.unique(y_train)) < 2:
        print(f"{target}: skipped — training labels lack both classes")
        continue
    if len(y_val) == 0 or len(np.unique(y_val)) < 2:
        print(f"{target}: AUC unavailable — validation lacks both classes")
        continue

    train_positions = np.flatnonzero(train_idx[train_mask])
    val_positions = np.flatnonzero(val_idx[val_mask])

    model = LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
    model.fit(X_train[train_positions], y_train)
    probabilities = model.predict_proba(X_val[val_positions])[:, 1]

    results.append({
        "target": target,
        "validation_positive": int(y_val.sum()),
        "validation_negative": int((y_val == 0).sum()),
        "auc": float(roc_auc_score(y_val, probabilities))
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

if not results_df.empty:
    macro_auc = results_df["auc"].mean()
    print(f"\nEXP-006 image + text macro-AUC: {macro_auc:.4f}")
    print(f"EXP-004 image-only: 0.5717")
    print(f"EXP-005 text-only: 0.5351")
    print(f"Difference vs image-only: {macro_auc - 0.5717:+.4f}")

results_df.to_csv(CACHE / "exp006_combined_results.csv", index=False)
print(f"\nSaved to {CACHE / 'exp006_combined_results.csv'}")


## 9. EXP-007 — Candidate weak-label generation

**GPU OFF.** This self-contained cell contains the current rule-based candidate-label script. It excludes all 58 fully labelled studies and does not treat silence as a negative. Outputs are written to `/kaggle/working/exp007_weak_labels/`.

**Important:** generating labels is not validating them. Inspect `weak_label_audit.csv` and `weak_label_summary.csv` before using any weak labels for training. In the earlier run, several targets had very imbalanced candidate coverage, and some targets had almost no candidate negatives.

In [ ]:
#!/usr/bin/env python3
"""Conservative rule-based weak labels from knee MRI reports.

Only studies with every official target missing are processed. Explicit positive
or negative cues may yield 1 or 0; no cue, uncertainty, or conflicting cues yield
NaN. This is a candidate generator, not validated clinical NLP.
"""
import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Keep these rules conservative. Silence is never a negative label.
RULES = {
"ACL": {
"positive": [
r"\b(?:acl|anterior cruciate ligament)\b.{0,40}\b(?:tear|torn|rupture|ruptured|sprain|injury)\b",
r"\b(?:tear|rupture|sprain|injury)\b.{0,40}\b(?:acl|anterior cruciate ligament)\b",
r"\b(?:rotura|ruptura)\b.{0,40}\bligamento cruzado anterior\b",
r"\b(?:rupture|déchirure)\b.{0,40}\bligament croisé antérieur\b",
r"\bvoorste kruisband\b.{0,30}\b(?:scheur|ruptuur|letsel)\b",
r"\b(?:acl|anterior cruciate ligament)\b.{0,100}\b(?:high[- ]grade|low[- ]grade|partial|complete)(?: thickness)?\s+tear\b",
r"\b(?:high[- ]grade|low[- ]grade|partial|complete)(?: thickness)?\s+tear\b.{0,100}\b(?:acl|anterior cruciate ligament)\b"],
"negative": [
r"\b(?:acl|anterior cruciate ligament)\b.{0,25}\b(?:intact|normal|unremarkable)\b",
r"\b(?:no|without|absence of)\b.{0,40}\b(?:acl|anterior cruciate ligament)\b.{0,25}\b(?:tear|rupture|injury)\b",
r"\bligamento cruzado anterior\b.{0,30}\b(?:íntegro|integro|normal)\b",
r"\bligament croisé antérieur\b.{0,30}\b(?:intact|normal)\b"]},
"MCL": {
"positive": [
r"\b(?:mcl|medial collateral ligament)\b.{0,40}\b(?:tear|rupture|sprain|injury|thickening)\b",
r"\b(?:tear|rupture|sprain|injury)\b.{0,40}\b(?:mcl|medial collateral ligament)\b",
r"\b(?:esguince|rotura|ruptura)\b.{0,35}\bligamento colateral medial\b",
r"\b(?:entorse|rupture|déchirure)\b.{0,35}\bligament collatéral médial\b"],
"negative": [
r"\b(?:mcl|medial collateral ligament)\b.{0,25}\b(?:intact|normal|unremarkable)\b",
r"\b(?:no|without|absence of)\b.{0,40}\b(?:mcl|medial collateral ligament)\b.{0,25}\b(?:tear|rupture|injury)\b",
r"\bligamento colateral medial\b.{0,25}\b(?:íntegro|integro|normal)\b"]},
"Medial Meniscus": {
"positive": [
r"\bmedial menisc(?:us|al)\b.{0,40}\b(?:tear|torn|rupture|flap)\b",
r"\b(?:tear|torn|rupture)\b.{0,40}\bmedial menisc(?:us|al)\b",
r"\b(?:rotura|ruptura)\b.{0,35}\bmenisco (?:medial|interno)\b",
r"\b(?:déchirure|rupture)\b.{0,35}\b(?:ménisque médial|ménisque interne)\b",
r"\b(?:mediale meniscus|binnenmeniscus)\b.{0,30}\b(?:scheur|ruptuur)\b"],
"negative": [
r"\bmedial menisc(?:us|al)\b.{0,30}\b(?:intact|normal|not torn|no tear|without tear)\b",
r"\b(?:no|without|absence of|no evidence of|no signs of)\b.{0,35}\b(?:tear|rupture)\b.{0,30}\bmedial menisc(?:us|al)\b",
r"\b(?:no|without|absence of|no evidence of|no signs of)\b.{0,35}\bmedial menisc(?:us|al)\b.{0,30}\b(?:tear|torn|rupture)\b",
r"\bmenisco (?:medial|interno)\b.{0,30}\b(?:sin rotura|normal|conservado)\b",
r"\b(?:ménisque médial|ménisque interne)\b.{0,30}\b(?:intact|normal|sans déchirure)\b"]},
"Lateral Meniscus": {
"positive": [
r"\blateral menisc(?:us|al)\b.{0,40}\b(?:tear|torn|rupture|flap)\b",
r"\b(?:tear|torn|rupture)\b.{0,40}\blateral menisc(?:us|al)\b",
r"\b(?:rotura|ruptura)\b.{0,35}\bmenisco lateral\b",
r"\b(?:déchirure|rupture)\b.{0,35}\b(?:ménisque latéral|ménisque externe)\b",
r"\b(?:laterale meniscus|buitenmeniscus)\b.{0,30}\b(?:scheur|ruptuur)\b"],
"negative": [
r"\blateral menisc(?:us|al)\b.{0,30}\b(?:intact|normal|not torn|no tear|without tear)\b",
r"\b(?:no|without|absence of|no evidence of|no signs of)\b.{0,35}\b(?:tear|rupture)\b.{0,30}\blateral menisc(?:us|al)\b",
r"\b(?:no|without|absence of|no evidence of|no signs of)\b.{0,35}\blateral menisc(?:us|al)\b.{0,30}\b(?:tear|torn|rupture)\b",
r"\bmenisco lateral\b.{0,30}\b(?:sin rotura|normal|conservado)\b",
r"\b(?:ménisque latéral|ménisque externe)\b.{0,30}\b(?:intact|normal|sans déchirure)\b"]},
"Medial OA": {
"positive": [
r"\bmedial (?:femorotibial |tibiofemoral )?compartment\b.{0,35}\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b",
r"\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b.{0,35}\bmedial (?:femorotibial |tibiofemoral )?compartment\b",
r"\bartrosis femorotibial medial\b",
r"\bartrosis (?:del )?compartimento medial\b",
r"\bcompartiment médial\b.{0,25}\b(?:arthrose|ostéoarthrose)\b"],
"negative": [
r"\bno\b.{0,25}\bmedial (?:femorotibial |tibiofemoral )?compartment (?:osteoarthritis|osteoarthrosis|arthrosis)\b",
r"\bpas d['’]arthrose\b.{0,35}\b(?:compartiment médial|compartimento medial)\b"]},
"Lateral OA": {
"positive": [
r"\blateral (?:femorotibial |tibiofemoral )?compartment\b.{0,35}\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b",
r"\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b.{0,35}\blateral (?:femorotibial |tibiofemoral )?compartment\b",
r"\bartrosis femorotibial lateral\b",
r"\bartrosis (?:del )?compartimento lateral\b",
r"\bcompartiment latéral\b.{0,25}\b(?:arthrose|ostéoarthrose)\b"],
"negative": [
r"\bno\b.{0,25}\blateral (?:femorotibial |tibiofemoral )?compartment (?:osteoarthritis|osteoarthrosis|arthrosis)\b",
r"\bpas d['’]arthrose\b.{0,35}\b(?:compartiment latéral|compartimento lateral)\b"]},
"PF OA": {
"positive": [
r"\bpatellofemoral\b.{0,35}\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose)\b",
r"\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose)\b.{0,35}\bpatellofemoral\b",
r"\b(?:retropatellar|patellofemoral)\b.{0,35}\b(?:arthrose|artrosis|osteoarthritis)\b"],
"negative": [
r"\bno\b.{0,25}\bpatellofemoral (?:osteoarthritis|osteoarthrosis|arthrosis)\b",
r"\bpas d['’]arthrose\b.{0,35}\b(?:patellofémoral|rétropatellaire)\b"]},
"Effusion": {
"positive": [r"\b(?:joint |knee |articular )?effusion\b",
r"\b(?:derrame articular|derrame de rodilla|épanchement articulaire|épanchement du genou|gewrichtseffusie)\b"],
"negative": [r"\b(?:no|without|absence of|no significant)\b.{0,25}\b(?:joint |knee |articular )?effusion\b",
r"\bno hay (?:evidente )?derrame(?: articular)?\b",
r"\bpas d['’]épanchement(?: articulaire)?\b", r"\bgeen (?:gewrichts)?effusie\b", r"\bkein gelenkerguss\b"]},
"Synovitis": {
"positive": [r"\bsynovitis\b", r"\bsinovitis\b", r"\bsynovite\b", r"\bsynoviale ontsteking\b"],
"negative": [r"\b(?:no|without|absence of)\b.{0,20}\bsynovitis\b", r"\bno hay (?:signos de )?sinovitis\b",
r"\bsin sinovitis\b", r"\bpas de synovite\b", r"\bgeen synovitis\b"]},
"Baker's": {
"positive": [r"\bbaker['’]?s? cyst\b", r"\bpopliteal cyst\b", r"\bquiste de baker\b", r"\bkyste poplité\b", r"\bcyste poplité\b", r"\bbakerzyste\b"],
"negative": [r"\b(?:no|without|absence of)\b.{0,25}\b(?:baker['’]?s? cyst|popliteal cyst)\b",
r"\bno hay (?:quiste de baker|quistes poplíteos?)\b", r"\bpas de kyste poplité\b"]},
"Contusion": {
"positive": [r"\bbone contusion\b", r"\bbone bruise\b", r"\bcontusión ósea\b", r"\bcontusion osseuse\b", r"\bknochenprellung\b"],
"negative": [r"\b(?:no|without|absence of)\b.{0,25}\b(?:bone contusion|bone bruise)\b",
r"\bno hay contusión ósea\b", r"\bpas de contusion osseuse\b", r"\bkeine knochenprellung\b"]},
"Fracture": {
"positive": [r"\bfracture\b", r"\bfractura\b", r"\bfractuur\b", r"\bfraktur\b"],
"negative": [r"\b(?:no|without|absence of|no evidence of)\b.{0,25}\bfracture\b",
r"\b(?:fractures?|fracturas?)\s*:\s*(?:none|no|aucune|ninguna)\b",
r"\bno hay (?:evidencia de )?fractura\b", r"\bpas de fracture\b", r"\bkeine fraktur\b", r"\bgeen fractuur\b"]},
}

UNCERTAINTY = re.compile(
    r"\b(?:rule\s+out|r/o|to exclude|possible|possibly|probable|probably|suspected|"
    r"suspicious for|cannot exclude|may represent|might represent|questionable|"
    r"indeterminate|posible|posiblemente|sospecha de|no se puede descartar|"
    r"suspect|à confirmer|mogelijk|verdacht voor)\b",
    re.IGNORECASE,
)
COMPILED = {
    t: {p: [re.compile(x, re.IGNORECASE) for x in pats] for p, pats in groups.items()}
    for t, groups in RULES.items()
}

def classify_report(report, target):
    raw = str(report or "")

    # Exclude clinical indication/history sections when a later report section
    # is present. This prevents suspected diagnoses from becoming evidence.
    raw = re.sub(
        r"(?is)\b(?:clinical\s+)?(?:indication|clinical information|clinical history|"
        r"history|reason for exam|clinical data)\s*:\s*.*?"
        r"(?=\b(?:findings?|impression|conclusion|report|technique|comparison)\s*:|\Z)",
        " ",
        raw,
    )

    # Split before whitespace normalisation so sentence/newline boundaries remain.
    clauses = [
        re.sub(r"\s+", " ", part).strip()
        for part in re.split(r"(?<=[.;!?])\s+|\n+", raw)
        if part.strip()
    ]
    spans = {"positive": [], "negative": []}
    uncertain = False

    for clause_index, clause in enumerate(clauses):
        remainder_clause = bool(
            re.match(r"^(?:the\s+)?remainder of\b", clause, re.IGNORECASE)
        )

        for polarity in ("positive", "negative"):
            for pattern in COMPILED[target][polarity]:
                for match in pattern.finditer(clause):
                    cue = match.group(0).strip()
                    context = clause[
                        max(0, match.start() - 100):
                        min(len(clause), match.end() + 100)
                    ]

                    if UNCERTAINTY.search(context):
                        uncertain = True
                        continue

                    # Loss of normal signal is abnormal ACL wording.
                    if polarity == "negative" and target == "ACL":
                        if re.search(
                            r"\b(?:loss of normal|partial|attenuated|attenuation|"
                            r"increased t2 signal|thinning)\b",
                            clause,
                            re.IGNORECASE,
                        ):
                            continue

                    # The remainder being intact does not negate a tear elsewhere.
                    if (
                        polarity == "negative"
                        and target in ("Medial Meniscus", "Lateral Meniscus")
                        and remainder_clause
                    ):
                        continue

                    spans[polarity].append(
                        (clause_index, match.start(), match.end(), cue)
                    )

    negative_spans = [
        (clause, start, end)
        for clause, start, end, _ in spans["negative"]
    ]
    positive = [
        cue
        for clause, start, end, cue in spans["positive"]
        if not any(
            clause == neg_clause
            and start < neg_end
            and neg_start < end
            for neg_clause, neg_start, neg_end in negative_spans
        )
    ]
    negative = [cue for _, _, _, cue in spans["negative"]]
    positive = list(dict.fromkeys(positive))
    negative = list(dict.fromkeys(negative))

    if positive and negative:
        return np.nan, "conflict", " | ".join(positive), " | ".join(negative)
    if positive:
        return 1.0, "positive", " | ".join(positive), ""
    if negative:
        return 0.0, "negative", "", " | ".join(negative)
    if uncertain:
        return np.nan, "uncertain", "", ""
    return np.nan, "unknown", "", ""


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-csv", default="/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv")
    parser.add_argument("--output-dir", default="/kaggle/working/exp007_weak_labels")
    args, _ = parser.parse_known_args()
    df = pd.read_csv(args.input_csv)
    required = {"StudyInstanceUID", "Report", *TARGETS}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if df["StudyInstanceUID"].duplicated().any():
        raise ValueError("StudyInstanceUID must be unique")
    if df["Report"].isna().any():
        raise ValueError("Report contains missing values; inspect source first")

    # Exclude every study with any official label. Do not touch gold labels.
    has_gold = df[TARGETS].notna().any(axis=1)
    weak_source = df.loc[~has_gold, ["StudyInstanceUID", "Report"]]
    print(f"Total studies: {len(df)}")
    print(f"Studies with official labels excluded: {int(has_gold.sum())}")
    print(f"Unlabelled studies to process: {len(weak_source)}")

    wide, audit = [], []
    for study_id, report in weak_source[["StudyInstanceUID", "Report"]].itertuples(index=False, name=None):
        output = {"StudyInstanceUID": study_id}
        for target in TARGETS:
            label, status, positive_cues, negative_cues = classify_report(report, target)
            output[target] = label
            audit.append({
                "StudyInstanceUID": study_id, "target": target, "weak_label": label,
                "status": status, "positive_cues": positive_cues, "negative_cues": negative_cues,
            })
        wide.append(output)

    outdir = Path(args.output_dir)
    outdir.mkdir(parents=True, exist_ok=True)
    weak_labels = pd.DataFrame(wide, columns=["StudyInstanceUID", *TARGETS])
    audit_df = pd.DataFrame(audit)
    weak_labels.to_csv(outdir / "weak_labels.csv", index=False)
    audit_df.to_csv(outdir / "weak_label_audit.csv", index=False)
    summary = audit_df.groupby(["target", "status"], dropna=False).size().rename("studies").reset_index()
    summary.to_csv(outdir / "weak_label_summary.csv", index=False)

    coverage = audit_df.groupby("target").agg(
        known=("weak_label", lambda s: int(s.notna().sum())),
        positive=("weak_label", lambda s: int((s == 1).sum())),
        negative=("weak_label", lambda s: int((s == 0).sum())),
        unknown=("status", lambda s: int((s == "unknown").sum())),
        uncertain=("status", lambda s: int((s == "uncertain").sum())),
        conflict=("status", lambda s: int((s == "conflict").sum())),
    )
    print("\nWeak-label coverage by target:")
    print(coverage.to_string())
    print(f"\nSaved outputs to {outdir}")
    print("Audit samples and error rates before using these labels for training.")



## 10. EXP-007 parser regression tests

**GPU OFF.** Run this cell after the EXP-007 definitions cell and before generating a new audit. It checks known parser failure cases without reading the dataset or writing output files.


In [ ]:
import math

REGRESSION_CASES = [
    (
        "Indication tear, findings deny tear",
        "Medial Meniscus",
        "Indication: suspected medial meniscal tear. "
        "Findings: Medial meniscus normal. "
        "Impression: No evidence of medial meniscal tear.",
        0.0,
        "negative",
    ),
    (
        "Tear with remainder intact",
        "Lateral Meniscus",
        "Findings: Tiny tear of the lateral meniscus. "
        "The remainder of the lateral meniscus is intact. "
        "Conclusion: Lateral meniscal tear.",
        1.0,
        "positive",
    ),
    (
        "ACL loss of normal fibres with high-grade tear",
        "ACL",
        "Findings: ACL: Loss of normal fiber signal and alignment "
        "with suspicious residual thin fiber, high grade tear.",
        1.0,
        "positive",
    ),
    (
        "Patellar facet OA is not lateral compartment OA",
        "Lateral OA",
        "Osteoarthritis of the lateral patellar facet.",
        float("nan"),
        "unknown",
    ),
    (
        "Explicit lateral femorotibial OA",
        "Lateral OA",
        "There is osteoarthritis of the lateral femorotibial compartment.",
        1.0,
        "positive",
    ),
    (
        "Contradictory synovitis statements",
        "Synovitis",
        "Findings: No hay signos de sinovitis. "
        "Impression: sinovitis difusa.",
        float("nan"),
        "conflict",
    ),
    (
        "Rule out tear without confirmation",
        "Medial Meniscus",
        "Impression: Rule out medial meniscal tear.",
        float("nan"),
        "uncertain",
    ),
]

REGRESSION_FAILURES = []
for case_name, target_name, report_text, expected_label, expected_status in REGRESSION_CASES:
    actual_label, actual_status, positive_cues, negative_cues = classify_report(
        report_text, target_name
    )
    label_matches = (
        math.isnan(actual_label)
        if math.isnan(expected_label)
        else actual_label == expected_label
    )
    passed = label_matches and actual_status == expected_status
    print(
        f"{'PASS' if passed else 'FAIL'}: {case_name} "
        f"| label={actual_label} | status={actual_status}"
    )
    if not passed:
        REGRESSION_FAILURES.append(
            (case_name, actual_label, actual_status, positive_cues, negative_cues)
        )

print(f"\n{len(REGRESSION_CASES) - len(REGRESSION_FAILURES)}/{len(REGRESSION_CASES)} tests passed")
assert not REGRESSION_FAILURES, REGRESSION_FAILURES


## 11. EXP-007 — Generate candidate weak-label audit

**GPU OFF.** Run this cell only after all regression tests pass. It processes the 4,349 studies without official labels and writes candidate labels plus an audit to /kaggle/working/exp007_weak_labels/. These outputs are temporary runtime files.


In [ ]:
main()


## 12. EXP-008 — Generate and validate submission.csv

**GPU ON.** This end-to-end submission cell extracts DINOv2 features from test DICOMs using the same series-selection, slice-sampling, and intensity-scaling policy as EXP-004, fits one logistic-regression classifier per target on all 58 officially labelled studies, and writes /kaggle/working/submission.csv.

The test reports are not read. The cell intentionally fails on missing eligible series, invalid features, incomplete gold labels, or malformed predictions rather than silently producing a partial submission. Run it with the competition dataset and the same permitted local pretrained DINOv2 model attached.

In [ ]:
# EXP-008 — final inference and submission validation.
# GPU ON. No test reports are loaded or used.
from pathlib import Path
import gc
import json
import time

import numpy as np
import pandas as pd
import pydicom
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
MODEL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/base/1"
CACHE = Path("/kaggle/working/exp004_dinov2_cache")
FEATURE_OUT = Path("/kaggle/working/exp008_test_features")
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
FEATURE_OUT.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
N_SLICES = 9
SEED = 42

# Load only test identifiers and series metadata; test reports are not used.
test_df = pd.read_csv(DATA / "test.csv")
test_series_df = pd.read_csv(DATA / "test_series.csv")
assert "StudyInstanceUID" in test_df.columns
assert "StudyInstanceUID" in test_series_df.columns
assert test_df["StudyInstanceUID"].notna().all()
assert test_df["StudyInstanceUID"].is_unique
assert len(test_df) > 0

# Load cached, officially labelled training features and labels.
train_features_3d = np.load(CACHE / "features.npy")
train_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True).astype(str)
gold_labels = pd.read_csv(CACHE / "labels.csv")
assert train_features_3d.shape == (58, N_SLICES, 768), train_features_3d.shape
assert len(train_ids) == len(gold_labels) == 58
assert len(set(train_ids)) == 58
assert gold_labels["StudyInstanceUID"].astype(str).tolist() == train_ids.tolist()
assert gold_labels[TARGETS].notna().all().all(), "Gold-label cache has missing targets."
assert np.isfinite(train_features_3d).all(), "Training feature cache contains non-finite values."

# Reload explicitly because earlier experiment cells reuse generic model variables.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("GPU is not enabled. Turn the Kaggle accelerator on before EXP-008.")

gc.collect()
torch.cuda.empty_cache()
processor = AutoImageProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
image_model = AutoModel.from_pretrained(MODEL_PATH, local_files_only=True).to(device).eval()
print("EXP-008 device:", device)
print("Test studies:", len(test_df), "| labelled training studies:", len(gold_labels))

def choose_test_series(study_id):
    candidates = test_series_df[test_series_df["StudyInstanceUID"].astype(str) == str(study_id)]
    sagittal = candidates[
        (candidates["Anatomical_Plane"] == "Sagittal")
        & (pd.to_numeric(candidates["Fluid_Sensitive"], errors="coerce") == 1)
    ]
    coronal = candidates[
        (candidates["Anatomical_Plane"] == "Coronal")
        & (pd.to_numeric(candidates["Fluid_Sensitive"], errors="coerce") == 1)
    ]
    if not sagittal.empty:
        return sagittal.sort_values("SeriesInstanceUID", kind="stable").iloc[0], "sagittal_fluid"
    if not coronal.empty:
        return coronal.sort_values("SeriesInstanceUID", kind="stable").iloc[0], "coronal_fluid_fallback"
    raise RuntimeError(f"No eligible fluid-sensitive sagittal/coronal series for test study {study_id}")

def load_test_dicom_paths(study_id, series_id):
    folder = DATA / "test_series" / str(study_id) / str(series_id)
    paths = list(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM files in {folder}")
    entries = []
    for path in paths:
        ds = pydicom.dcmread(path, stop_before_pixels=True)
        instance = getattr(ds, "InstanceNumber", None)
        if instance is None:
            raise ValueError(f"Missing InstanceNumber: {path}")
        entries.append((int(instance), path))
    entries.sort(key=lambda item: (item[0], item[1].name))
    instances = [item[0] for item in entries]
    if len(instances) != len(set(instances)):
        raise ValueError(f"Duplicate InstanceNumber in {study_id}/{series_id}")
    return [item[1] for item in entries]

def test_dicom_to_image(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1))
    pixels += float(getattr(ds, "RescaleIntercept", 0))
    lo, hi = np.percentile(pixels, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        raise ValueError(f"Invalid intensity range: {path}")
    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
    pixels = (pixels * 255).astype(np.uint8)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = 255 - pixels
    return Image.fromarray(pixels).convert("RGB")

# Extract nine deterministic, evenly spaced slices per test study.
test_feature_rows = []
feature_metadata = []
start_time = time.time()
test_ids = test_df["StudyInstanceUID"].astype(str).tolist()

for study_idx, study_id in enumerate(test_ids, start=1):
    chosen, selection = choose_test_series(study_id)
    series_id = str(chosen["SeriesInstanceUID"])
    paths = load_test_dicom_paths(study_id, series_id)
    indices = np.rint(np.linspace(0, len(paths) - 1, N_SLICES)).astype(int)
    if len(set(indices.tolist())) != N_SLICES:
        raise RuntimeError(
            f"Selected series has too few slices for {N_SLICES} unique samples: "
            f"{study_id}/{series_id}, slices={len(paths)}"
        )

    images = [test_dicom_to_image(paths[int(idx)]) for idx in indices]
    inputs = processor(images=images, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.inference_mode():
        outputs = image_model(**inputs)
        feats = outputs.last_hidden_state[:, 0, :].float().cpu().numpy()

    if feats.shape != (N_SLICES, 768) or not np.isfinite(feats).all():
        raise RuntimeError(f"Invalid test features for {study_id}: shape={feats.shape}")
    test_feature_rows.append(feats)
    feature_metadata.append({
        "StudyInstanceUID": study_id,
        "SeriesInstanceUID": series_id,
        "selection": selection,
        "series_slice_count": len(paths),
        "sampled_slice_indices": ",".join(map(str, indices.tolist()))
    })

    if study_idx % 25 == 0 or study_idx == len(test_ids):
        elapsed = time.time() - start_time
        print(f"{study_idx}/{len(test_ids)} test studies | {elapsed:.1f}s elapsed")

test_features_3d = np.stack(test_feature_rows).astype(np.float32)
assert test_features_3d.shape == (len(test_df), N_SLICES, 768)
assert np.isfinite(test_features_3d).all()
np.save(FEATURE_OUT / "test_features.npy", test_features_3d)
pd.DataFrame(feature_metadata).to_csv(FEATURE_OUT / "test_feature_metadata.csv", index=False)
print("Test feature shape:", test_features_3d.shape)
print("Test feature extraction seconds:", round(time.time() - start_time, 2))

# Match EXP-004: average the nine per-slice CLS embeddings into one study vector.
X_train = train_features_3d.mean(axis=1)
X_test = test_features_3d.mean(axis=1)
assert X_train.shape == (58, 768)
assert X_test.shape == (len(test_df), 768)

submission = pd.DataFrame({"StudyInstanceUID": test_ids})
model_summaries = []

for target in TARGETS:
    y = pd.to_numeric(gold_labels[target], errors="coerce").to_numpy()
    if not np.isfinite(y).all() or not set(np.unique(y)).issubset({0, 1}):
        raise ValueError(f"Invalid gold labels for {target}")
    if len(np.unique(y)) != 2:
        raise ValueError(f"Cannot fit binary classifier: only one class for {target}")

    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED
        )
    )
    classifier.fit(X_train, y.astype(int))
    probabilities = classifier.predict_proba(X_test)[:, 1]
    if probabilities.shape != (len(test_df),) or not np.isfinite(probabilities).all():
        raise RuntimeError(f"Invalid predictions for {target}")
    if ((probabilities < 0) | (probabilities > 1)).any():
        raise RuntimeError(f"Out-of-range probabilities for {target}")
    submission[target] = probabilities
    model_summaries.append({
        "target": target,
        "n_train": int(len(y)),
        "train_positive": int(y.sum()),
        "train_negative": int((y == 0).sum()),
        "prediction_min": float(probabilities.min()),
        "prediction_max": float(probabilities.max()),
        "prediction_mean": float(probabilities.mean())
    })

EXPECTED_COLUMNS = ["StudyInstanceUID"] + TARGETS
assert submission.columns.tolist() == EXPECTED_COLUMNS
assert len(submission) == len(test_df)
assert submission["StudyInstanceUID"].astype(str).tolist() == test_ids
assert submission["StudyInstanceUID"].is_unique
assert submission[TARGETS].notna().all().all()
assert np.isfinite(submission[TARGETS].to_numpy(dtype=np.float64)).all()
assert ((submission[TARGETS] >= 0) & (submission[TARGETS] <= 1)).all().all()

submission.to_csv(SUBMISSION_PATH, index=False)
# Read back the saved artifact and validate the actual CSV.
written = pd.read_csv(SUBMISSION_PATH)
assert written.columns.tolist() == EXPECTED_COLUMNS
assert len(written) == len(test_df)
assert written["StudyInstanceUID"].astype(str).tolist() == test_ids
assert np.isfinite(written[TARGETS].to_numpy(dtype=np.float64)).all()
assert ((written[TARGETS] >= 0) & (written[TARGETS] <= 1)).all().all()

pd.DataFrame(model_summaries).to_csv(FEATURE_OUT / "prediction_summary.csv", index=False)
manifest = {
    "experiment": "EXP-008",
    "status": "submission file generated and structurally validated; performance not yet measured",
    "feature_model": "DINOv2 Base local pretrained weights",
    "feature_dim": 768,
    "slices_per_study": N_SLICES,
    "series_selection": "fluid-sensitive sagittal; fluid-sensitive coronal fallback",
    "slice_sampling": "9 rounded evenly spaced indices after InstanceNumber ordering",
    "preprocessing": "per-slice 1st/99th percentile scaling, MONOCHROME1 inversion, RGB",
    "classifier": "per-target StandardScaler + LogisticRegression(C=1.0, class_weight=balanced, max_iter=2000)",
    "training_studies": int(len(gold_labels)),
    "test_studies": int(len(test_df)),
    "random_seed": SEED,
    "submission_path": str(SUBMISSION_PATH),
    "test_feature_extraction_seconds": round(time.time() - start_time, 2)
}
(FEATURE_OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("\nSubmission written:", SUBMISSION_PATH)
print("Submission shape:", written.shape)
print("Submission columns:", written.columns.tolist())
print("All predictions finite and within [0, 1]:", bool(np.isfinite(written[TARGETS].to_numpy()).all()))
print("Prediction summary:")
print(pd.DataFrame(model_summaries).to_string(index=False))
print("\nImportant: this is a generated candidate submission, not yet a measured leaderboard result.")


## 13. EXP-010 — Build and audit study-level out-of-fold folds

**GPU OFF.** This cell is independent of the model-training cells. It uses only the 58 studies with all 12 official labels, creates deterministic multi-label-aware fold assignments, checks that every fold has positive and negative examples for every target, and writes fold/support CSVs under /kaggle/working.

This is a new evaluation protocol; do not overwrite or relabel the historical SPLIT-42-ACL-43-15 results. A valid fold assignment is not itself a performance result.

In [ ]:
# EXP-010 — deterministic multi-label-aware study folds. GPU OFF.
from pathlib import Path
import numpy as np
import pandas as pd

DATA_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
TRAIN_CSV = DATA_ROOT / "train.csv"
FOLD_OUTPUT = Path("/kaggle/working/exp010_folds.csv")
SUPPORT_OUTPUT = Path("/kaggle/working/exp010_fold_support.csv")
N_SPLITS = 5
SEED = 42

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

def make_multilabel_folds(study_ids, labels, n_splits=5, seed=42):
    ids = np.asarray(study_ids, dtype=str)
    y = np.asarray(labels)
    if ids.ndim != 1 or y.ndim != 2 or y.shape[0] != len(ids) or y.shape[1] == 0:
        raise ValueError("Expected one label row per study and at least one target")
    if len(set(ids.tolist())) != len(ids):
        raise ValueError("Duplicate study IDs would risk leakage")
    if not np.isfinite(y).all() or not np.isin(y, [0, 1]).all():
        raise ValueError("Fold generation requires complete binary labels")
    if not isinstance(n_splits, int) or n_splits < 2 or len(ids) < n_splits:
        raise ValueError("Invalid number of folds")
    y = y.astype(np.int8, copy=False)
    positives = y.sum(axis=0)
    negatives = len(ids) - positives
    bad = [j for j, (p, n) in enumerate(zip(positives, negatives))
           if p < n_splits or n < n_splits]
    if bad:
        raise ValueError(
            f"Targets at column indices {bad} lack at least {n_splits} positives "
            "and negatives; reduce folds only after inspecting and documenting support."
        )

    rng = np.random.default_rng(seed)
    desired = np.tile(positives / float(n_splits), (n_splits, 1))
    sizes = np.zeros(n_splits, dtype=np.int64)
    folds = np.full(len(ids), -1, dtype=np.int64)
    remaining = set(range(len(ids)))
    while remaining:
        remaining_idx = np.fromiter(sorted(remaining), dtype=np.int64)
        counts = y[remaining_idx].sum(axis=0)
        active = np.flatnonzero(counts > 0)
        if len(active) == 0:
            for i in rng.permutation(remaining_idx):
                choices = np.flatnonzero(sizes == sizes.min())
                fold = int(rng.choice(choices))
                folds[i] = fold
                sizes[fold] += 1
                remaining.remove(int(i))
            break
        rare = active[counts[active] == counts[active].min()]
        label = int(rng.choice(rare))
        candidates_studies = np.array(
            [i for i in remaining_idx if y[i, label] == 1], dtype=np.int64
        )
        rng.shuffle(candidates_studies)
        for i in candidates_studies:
            if int(i) not in remaining:
                continue
            best = desired[:, label].max()
            choices = np.flatnonzero(np.isclose(desired[:, label], best))
            choices = choices[sizes[choices] == sizes[choices].min()]
            fold = int(rng.choice(choices))
            folds[i] = fold
            sizes[fold] += 1
            desired[fold] -= y[i]
            remaining.remove(int(i))

    if (folds < 0).any() or set(folds.tolist()) != set(range(n_splits)):
        raise RuntimeError("Not every fold received studies")
    for fold in range(n_splits):
        yf = y[folds == fold]
        if len(yf) == 0 or (yf.sum(axis=0) == 0).any() or (yf.sum(axis=0) == len(yf)).any():
            raise ValueError(
                f"Fold {fold} lacks positive or negative support for a target; "
                "inspect the counts and reduce n_splits if necessary."
            )
    return folds

if not TRAIN_CSV.is_file():
    raise FileNotFoundError(f"Competition train.csv not found: {TRAIN_CSV}")

train_df = pd.read_csv(TRAIN_CSV)
required = ["StudyInstanceUID", *LABELS]
missing = [col for col in required if col not in train_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

gold = train_df.dropna(subset=LABELS).copy()
if gold["StudyInstanceUID"].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID in complete-label studies")
fold_ids = make_multilabel_folds(
    gold["StudyInstanceUID"].astype(str).tolist(),
    gold[LABELS].to_numpy(),
    n_splits=N_SPLITS,
    seed=SEED,
)

fold_df = gold[["StudyInstanceUID"]].copy()
fold_df["fold"] = fold_ids
FOLD_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fold_df.to_csv(FOLD_OUTPUT, index=False)

support_rows = []
for fold in range(N_SPLITS):
    part = gold.loc[fold_ids == fold, LABELS]
    row = {"fold": fold, "n_studies": int(len(part))}
    for label in LABELS:
        row[f"{label}__positive"] = int((part[label] == 1).sum())
        row[f"{label}__negative"] = int((part[label] == 0).sum())
    support_rows.append(row)
support_df = pd.DataFrame(support_rows)
support_df.to_csv(SUPPORT_OUTPUT, index=False)

assert len(fold_df) == len(gold)
assert fold_df["StudyInstanceUID"].is_unique
assert set(fold_df["fold"]) == set(range(N_SPLITS))
print(f"Complete-label studies: {len(gold)}")
print(f"Seed: {SEED} | folds: {N_SPLITS}")
print("\nPer-fold label support:")
for row in support_rows:
    print(f"Fold {row['fold']}: n={row['n_studies']}")
    for label in LABELS:
        positive = row[label + "__positive"]
        negative = row[label + "__negative"]
        print(f"  {label}: +{positive} / -{negative}")
print(f"\nSaved fold assignments: {FOLD_OUTPUT}")
print(f"Saved support audit: {SUPPORT_OUTPUT}")
print("This cell only creates and audits folds; no model was trained.");


In [ ]:
# ===== EXP-010 fold-integrity audit =====
# Kaggle: GPU OFF. NEW cell, run once. No model, no DINOv2, no training.
# Needs only train.csv and /kaggle/working/exp010_folds.csv (saved by the earlier EXP-010 cell).
import hashlib, json, os, platform, sys, urllib.request
import numpy as np
import pandas as pd

TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
FOLDS_CSV = "/kaggle/working/exp010_folds.csv"
OUT_JSON  = "/kaggle/working/exp010_fold_audit.json"
N_SPLITS, SEED, EXPECTED_COMPLETE = 5, 42, 58
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
# Support table you pasted from the earlier EXP-010 run (positives per label, same label order)
REPORTED = {0: (13, [4,1,5,4,3,3,5,6,5,3,5,3]), 1: (10, [4,2,5,4,3,2,4,6,6,2,4,3]),
            2: (12, [5,2,6,4,3,2,4,6,5,3,3,3]), 3: (12, [4,2,4,5,3,2,4,8,5,2,3,3]),
            4: (11, [7,2,6,6,3,2,4,9,6,2,4,6])}
REPO_RAW = "https://raw.githubusercontent.com/vixintry/rsna-knee-m1/notebook-cleanup/src/"
REPO_VALIDATION_BLOB = "baff60e34423f7de7f3bb66fcb69488a342bb5de"  # git blob SHA seen in the repo

results, notes = [], []
def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(("PASS  " if ok else "FAIL  ") + name + (f"  -- {detail}" if detail else ""))
def sha256(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()

# ---- 0. load inputs ----
train = pd.read_csv(TRAIN_CSV)
folds = pd.read_csv(FOLDS_CSV)
print("train.csv rows:", len(train), "| folds file columns:", list(folds.columns), "| rows:", len(folds))
id_col = "StudyInstanceUID"
fold_col = next((c for c in folds.columns if c.lower() in ("fold", "fold_id", "oof_fold")), None)
check("folds file has StudyInstanceUID and a fold column", id_col in folds.columns and fold_col is not None,
      f"fold column = {fold_col}")
if id_col not in folds.columns or fold_col is None:
    raise SystemExit("Cannot continue: unexpected fold-file columns (see above).")

# ---- 1. complete-label studies ----
check("train.csv has all 12 label columns + study id", all(c in train.columns for c in LABELS + [id_col]))
check("train.csv StudyInstanceUID unique", not train[id_col].duplicated().any())
complete = train.dropna(subset=LABELS).copy()
check(f"exactly {EXPECTED_COMPLETE} complete-label studies", len(complete) == EXPECTED_COMPLETE, f"found {len(complete)}")
check("complete-label values are only 0/1", complete[LABELS].isin([0, 1]).all().all())

# ---- 2. fold file vs complete-label studies (by ID, not row order) ----
check("no missing/blank study IDs in folds file", folds[id_col].notna().all() and (folds[id_col].astype(str).str.strip() != "").all())
check("every study appears exactly once in folds file", not folds[id_col].duplicated().any())
ids_f, ids_c = set(folds[id_col].astype(str)), set(complete[id_col].astype(str))
check("fold-file IDs == complete-label IDs exactly", ids_f == ids_c,
      f"in folds not complete: {len(ids_f-ids_c)}, in complete not folds: {len(ids_c-ids_f)}")
fv = folds[fold_col]
check("fold values are integers with no NaN", fv.notna().all() and np.all(np.equal(np.mod(fv.dropna(), 1), 0)))
fv = fv.astype(int)
check(f"fold IDs are exactly {{0..{N_SPLITS-1}}}", set(fv.unique()) == set(range(N_SPLITS)), f"found {sorted(int(x) for x in fv.unique())}")
sizes = fv.value_counts().sort_index()
check("all folds non-empty", len(sizes) == N_SPLITS and (sizes > 0).all(), f"sizes {sizes.to_dict()}")

# ---- 3. join labels by StudyInstanceUID (validate one-to-one) ----
try:
    merged = complete[[id_col] + LABELS].assign(**{id_col: complete[id_col].astype(str)}).merge(
        folds[[id_col, fold_col]].assign(**{id_col: folds[id_col].astype(str)}),
        on=id_col, how="inner", validate="one_to_one")
    check("ID join is one-to-one and covers all studies", len(merged) == len(complete) == len(folds))
except Exception as e:
    check("ID join is one-to-one and covers all studies", False, f"{type(e).__name__}")
    print("\nVERDICT: FAILED -- fix the integrity failures above before any training. Do NOT change the split algorithm until diagnosed.")
    raise SystemExit
merged = merged.rename(columns={fold_col: "fold"})

# ---- 4. per-label support per fold (from the ID-joined data) ----
print("\nPer-fold support (positives/negatives):")
sup_rows, low_support = [], []
for f in range(N_SPLITS):
    sub = merged[merged["fold"] == f]
    pos = sub[LABELS].sum().astype(int)
    neg = len(sub) - pos
    sup_rows.append({"fold": f, "n": len(sub), **{l: f"{pos[l]}/{neg[l]}" for l in LABELS}})
    for l in LABELS:
        if pos[l] < 2 or neg[l] < 2: low_support.append((f, l, int(pos[l]), int(neg[l])))
print(pd.DataFrame(sup_rows).to_string(index=False))
both = all((merged[merged["fold"] == f][LABELS].sum() > 0).all() and
           (merged[merged["fold"] == f][LABELS].sum() < (merged["fold"] == f).sum()).all() for f in range(N_SPLITS))
check("every label has both classes in every fold", both)
tot_pos = merged[LABELS].sum().astype(int)
check("every label has >=5 positives and >=5 negatives overall",
      ((tot_pos >= N_SPLITS) & ((len(merged) - tot_pos) >= N_SPLITS)).all(),
      f"min overall positives: {tot_pos.min()} ({tot_pos.idxmin()})")
if low_support:
    notes.append("Folds with <2 positives or <2 negatives for a label (per-fold AUC for these is close to meaningless; "
                 f"use pooled OOF AUC): {low_support}")
# cross-check against the table reported earlier (informational: confirms THIS file made that table)
rep_ok = True
for f, (n, pos_list) in REPORTED.items():
    sub = merged[merged["fold"] == f]
    rep_ok &= (len(sub) == n) and (sub[LABELS].sum().astype(int).tolist() == pos_list)
print(("\nCROSS-CHECK  " if rep_ok else "\nCROSS-CHECK MISMATCH  ") +
      "saved fold file " + ("matches" if rep_ok else "does NOT match") + " the support table from the earlier run")
check("saved fold file matches the previously reported support table", rep_ok)

# ---- 5. regenerate with same seed: determinism + reproduction of the saved file ----
candidates = []
if "make_multilabel_folds" in globals():
    candidates.append(("notebook global make_multilabel_folds (source unverified)", globals()["make_multilabel_folds"]))
try:
    from src.validation import make_multilabel_folds as _f
    candidates.append(("src.validation (already importable)", _f))
except Exception:
    try:
        os.makedirs("/kaggle/working/_repo_src/src", exist_ok=True)
        for fn in ("__init__.py", "labels.py", "validation.py"):
            data = urllib.request.urlopen(REPO_RAW + fn, timeout=30).read()
            open(f"/kaggle/working/_repo_src/src/{fn}", "wb").write(data)
            if fn == "validation.py":
                blob = hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest()
                notes.append(f"downloaded src/validation.py git-blob SHA {blob} "
                             + ("(matches the SHA I reviewed)" if blob == REPO_VALIDATION_BLOB else "(DIFFERENT from the reviewed SHA baff60e: file changed)"))
        sys.path.insert(0, "/kaggle/working/_repo_src")
        from src.validation import make_multilabel_folds as _f
        candidates.append(("repo src/validation.py (downloaded from GitHub)", _f))
    except Exception as e:
        notes.append(f"Could not obtain repo generator ({type(e).__name__}: {e}). Internet is probably off.")
check("at least one fold generator available for the rerun", len(candidates) > 0)

saved = dict(zip(folds[id_col].astype(str), fv))
orderings = {"train.csv row order": complete.reset_index(drop=True),
             "sorted by StudyInstanceUID": complete.sort_values(id_col).reset_index(drop=True)}
reproduced_any = False
for cname, fn in candidates:
    for oname, df in orderings.items():
        ids = df[id_col].astype(str).tolist(); y = df[LABELS].to_numpy()
        a = fn(ids, y, n_splits=N_SPLITS, seed=SEED); b = fn(ids, y, n_splits=N_SPLITS, seed=SEED)
        det = np.array_equal(a, b)
        same = all(saved[i] == int(f) for i, f in zip(ids, a))
        reproduced_any |= (det and same)
        print(f"  [{cname}] [{oname}] deterministic={det}  reproduces_saved_file={same}")
        check(f"deterministic rerun: {cname} / {oname}", det)
check("saved folds reproduced exactly by at least one (generator, ordering)", reproduced_any if candidates else False)
if candidates and not reproduced_any:
    notes.append("Saved folds were NOT reproduced by any available generator/ordering. Do NOT change the algorithm yet: "
                 "diagnose which code produced the file (see which source/ordering printed False).")
if candidates:
    notes.append("The generator indexes studies by row position, so folds depend on row order. The saved CSV, joined by "
                 "StudyInstanceUID, is the canonical artefact; never regenerate folds mid-project.")

# ---- 6. record + verdict ----
hard_fail = [n for n, ok, _ in results if not ok]
audit = {"train_csv_sha256": sha256(TRAIN_CSV), "folds_csv_sha256": sha256(FOLDS_CSV), "n_splits": N_SPLITS, "seed": SEED,
         "fold_sizes": {int(k): int(v) for k, v in sizes.items()}, "checks": [(n, ok, d) for n, ok, d in results],
         "notes": notes, "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__}
json.dump(audit, open(OUT_JSON, "w"), indent=2, default=str)
print("\n" + "=" * 60)
for n in notes: print("NOTE:", n)
print("folds_csv sha256:", audit["folds_csv_sha256"])
print("train_csv sha256:", audit["train_csv_sha256"])
print("saved audit ->", OUT_JSON)
print("VERDICT:", "ALL CHECKS PASSED" if not hard_fail else f"{len(hard_fail)} CHECK(S) FAILED: {hard_fail}")

In [ ]:
# ===== EXP-010: locate the missing fold file, or verifiably recover it =====
# Kaggle: GPU OFF. NEW cell, run once. No model, no training.
import glob, hashlib, json, os, sys, urllib.request
import numpy as np
import pandas as pd

TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
FOLDS_CSV = "/kaggle/working/exp010_folds.csv"
N_SPLITS, SEED = 5, 42
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
# Support table from your earlier EXP-010 run: fold -> (n studies, positives per label in LABELS order)
REPORTED = {0: (13, [4,1,5,4,3,3,5,6,5,3,5,3]), 1: (10, [4,2,5,4,3,2,4,6,6,2,4,3]),
            2: (12, [5,2,6,4,3,2,4,6,5,3,3,3]), 3: (12, [4,2,4,5,3,2,4,8,5,2,3,3]),
            4: (11, [7,2,6,6,3,2,4,9,6,2,4,6])}
REPO_RAW = "https://raw.githubusercontent.com/vixintry/rsna-knee-m1/notebook-cleanup/src/"
REPO_VALIDATION_BLOB = "baff60e34423f7de7f3bb66fcb69488a342bb5de"

# ---- 1. is the file anywhere? ----
print("exists at expected path:", os.path.exists(FOLDS_CSV))
print("/kaggle/working contents:", sorted(os.listdir("/kaggle/working")) if os.path.isdir("/kaggle/working") else "missing")
hits = []
for root in ("/kaggle/working", os.getcwd(), os.path.expanduser("~")):
    hits += glob.glob(os.path.join(root, "**", "*exp010*"), recursive=True)
print("files matching *exp010* :", sorted(set(hits)) or "none")
if os.path.exists(FOLDS_CSV):
    raise SystemExit("The file exists now -- rerun the audit cell instead of this one.")

# ---- 2. get the generator(s) ----
candidates, notes = [], []
if "make_multilabel_folds" in globals():
    candidates.append(("notebook global make_multilabel_folds (source unverified)", globals()["make_multilabel_folds"]))
if not candidates:
    try:
        os.makedirs("/kaggle/working/_repo_src/src", exist_ok=True)
        for fn in ("__init__.py", "labels.py", "validation.py"):
            data = urllib.request.urlopen(REPO_RAW + fn, timeout=30).read()
            out_path = f"/kaggle/working/_repo_src/src/{fn}"
            with open(out_path, "wb") as f:
                f.write(data)
            if fn == "validation.py":
                blob = hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest()
                if blob != REPO_VALIDATION_BLOB:
                    raise RuntimeError(
                        f"Refusing unreviewed validation.py: {blob}; "
                        f"expected {REPO_VALIDATION_BLOB}"
                    )
                notes.append(f"Downloaded validation.py has reviewed blob SHA {blob}")
        sys.path.insert(0, "/kaggle/working/_repo_src")
        from src.validation import make_multilabel_folds as _f
        candidates.append(("repo src/validation.py (hash verified)", _f))
    except Exception as e:
        notes.append(f"Could not obtain verified repo generator ({type(e).__name__}: {e})")
if not candidates:
    print(notes); raise SystemExit("No generator available; paste the error above to Claude.")

# ---- 3. regenerate IN MEMORY and compare with the table you reported ----
train = pd.read_csv(TRAIN_CSV)
complete = train.dropna(subset=LABELS).copy()
assert len(complete) == 58 and not complete["StudyInstanceUID"].duplicated().any()
orderings = {"train.csv row order": complete.reset_index(drop=True),
             "sorted by StudyInstanceUID": complete.sort_values("StudyInstanceUID").reset_index(drop=True)}
def table_matches(df, folds):
    ok = True
    for f, (n, pos) in REPORTED.items():
        sub = df[folds == f]
        ok &= (len(sub) == n) and (sub[LABELS].sum().astype(int).tolist() == pos)
    return bool(ok)
winner = None
for cname, fn in candidates:
    for oname, df in orderings.items():
        folds = fn(df["StudyInstanceUID"].astype(str).tolist(), df[LABELS].to_numpy(), n_splits=N_SPLITS, seed=SEED)
        m = table_matches(df, folds)
        print(f"[{cname}] [{oname}] fold sizes={np.bincount(folds).tolist()}  matches reported table: {m}")
        if m and winner is None: winner = (cname, oname, df, folds)

# ---- 4. only save if the regeneration reproduces your reported table exactly ----
for n in notes: print("NOTE:", n)
if winner is None:
    print("\nNO regeneration matched the reported table. Nothing was saved. Do not invent a replacement split; paste this output to Claude.")
else:
    cname, oname, df, folds = winner
    out = pd.DataFrame({"StudyInstanceUID": df["StudyInstanceUID"].astype(str), "fold": folds})
    out.to_csv(FOLDS_CSV, index=False)
    prov = {"status": "REGENERATED (original file was lost), not the original artefact",
            "generator": cname, "ordering": oname, "seed": SEED, "n_splits": N_SPLITS,
            "evidence": "per-fold sizes and per-label positive counts equal the table reported from the original run",
            "limit": "matching counts is strong evidence but not proof that fold membership is identical to the original",
            "folds_csv_sha256": hashlib.sha256(open(FOLDS_CSV, "rb").read()).hexdigest()}
    json.dump(prov, open("/kaggle/working/exp010_folds_PROVENANCE.json", "w"), indent=2)
    print("\nSaved", FOLDS_CSV, "| sha256", prov["folds_csv_sha256"])
    print("Provenance:", json.dumps(prov, indent=2))


### Step 1 — Validate Cached Features and Locate Fold Files

Inspect the existing DINOv2 cache and locate the previously audited fold files. No training or feature extraction.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Search only shallow paths; do not recursively scan the full DICOM dataset.
print("=== FOLD FILE SEARCH ===")
roots = [Path("/kaggle/working"), Path("/kaggle/input")]
found = []
for root in roots:
    if not root.exists():
        continue
    candidates = list(root.glob("*exp010*"))
    for directory in root.iterdir():
        if directory.is_dir():
            candidates.extend(directory.glob("*exp010*"))
    found.extend(p for p in candidates if p.is_file())

found = sorted(set(found))
if found:
    for path in found:
        print(path)
else:
    print("No exp010 fold CSV or audit JSON found.")

# Kaggle working files are temporary. Missing cache files should not prevent
# the fold-file search above from completing.
cache = Path("/kaggle/working/exp004_dinov2_cache")
required = [
    "features.npy", "study_ids.npy", "labels.csv",
    "manifest.json", "slice_metadata.csv", "split_manifest.csv",
]
missing = [name for name in required if not (cache / name).is_file()]

print("\n=== CACHE FILES ===")
for name in required:
    print(f"{name}: {'MISSING' if name in missing else 'EXISTS'}")

if missing:
    print("\nCACHE VALIDATION SKIPPED: temporary cache files are missing.")
    print("Do not rerun feature extraction just to locate the fold files.")
else:
    features = np.load(cache / "features.npy", mmap_mode="r")
    study_ids = np.load(cache / "study_ids.npy", allow_pickle=False).astype(str)
    labels = pd.read_csv(cache / "labels.csv")
    metadata = pd.read_csv(cache / "slice_metadata.csv")
    split_manifest = pd.read_csv(cache / "split_manifest.csv")

    print("\n=== SHAPES ===")
    print("Features:", features.shape)
    print("Study IDs:", study_ids.shape)
    print("Labels:", labels.shape)
    print("Slice metadata:", metadata.shape)
    print("Split manifest:", split_manifest.shape)

    if features.ndim != 3 or features.shape[0] != len(study_ids):
        raise ValueError("Feature array dimensions do not align with study IDs.")
    if not np.isfinite(features).all():
        raise ValueError("Cached features contain non-finite values.")
    if len(np.unique(study_ids)) != len(study_ids):
        raise ValueError("Duplicate study IDs in feature cache.")

    with open(cache / "manifest.json", encoding="utf-8") as f:
        manifest = json.load(f)
    print("\n=== CACHE MANIFEST ===")
    print(json.dumps(manifest, indent=2))

    id_columns = [
        c for c in labels.columns
        if c.lower() in {"studyinstanceuid", "study_id"}
    ]
    if not id_columns:
        raise ValueError("No recognised study-ID column in labels.csv.")

    label_ids = labels[id_columns[0]].astype(str)
    if not label_ids.is_unique or set(label_ids) != set(study_ids):
        raise ValueError("Feature IDs and label IDs do not match exactly.")

    print("Label ID column:", id_columns[0])
    print("Feature/label IDs align: True")
    print("All cached features finite: True")
    print("Note: split_manifest.csv is not the EXP-010 fold assignment.")



## 14. EXP-011 — Five-fold cross-validation for DINOv2 image features

**GPU OFF.** Evaluates the existing EXP-004 image-only logistic-regression baseline using the original, audited EXP-010 study-level folds.

**Method**
- Average the nine frozen DINOv2 slice embeddings into one 768-dimensional vector per study.
- Use the saved five-fold assignments without regenerating or modifying them.
- Train a separate one-vs-rest logistic regression model for each of the 12 targets in each fold.
- Apply `StandardScaler` within a training-only pipeline to avoid preprocessing leakage.
- Preserve EXP-004 settings: `C=1.0`, `class_weight="balanced"`, `max_iter=2000`, `random_state=42`.
- Generate one out-of-fold prediction per study and target.

**Evaluation**
- Report per-target ROC-AUC, per-fold macro-AUC, and overall out-of-fold macro-AUC.
- Save OOF predictions, fold metrics, and aggregate metrics under `/kaggle/working/`.
- Treat results as exploratory: only 58 fully labelled studies are available.

**Integrity rules:** study-level folds remain fixed; missing labels are never converted to negatives; no fold's validation labels are used to fit its model.


In [5]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ---------------- Configuration ----------------
CACHE = Path("/kaggle/working/exp004_dinov2_cache")
FOLD_PATH = Path(
    "/kaggle/input/datasets/maratiusupov/exp010-folds/exp010_folds.csv"
)
OUT = Path("/kaggle/working/exp011_cv_results")
OUT.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
ID_COL = "StudyInstanceUID"
SEED = 42
N_FOLDS = 5

# ---------------- Load and validate inputs ----------------
features = np.load(CACHE / "features.npy")
study_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True).astype(str)
labels = pd.read_csv(CACHE / "labels.csv")
folds = pd.read_csv(FOLD_PATH)

assert features.shape == (58, 9, 768), features.shape
assert len(study_ids) == 58 and len(set(study_ids)) == 58
assert labels[ID_COL].astype(str).is_unique
assert folds[ID_COL].astype(str).is_unique
assert set(labels[ID_COL].astype(str)) == set(study_ids)
assert set(folds[ID_COL].astype(str)) == set(study_ids)
assert set(folds["fold"]) == set(range(N_FOLDS))
assert labels[TARGETS].notna().all().all(), "Missing target labels"
assert np.isfinite(features).all(), "Non-finite features"

for target in TARGETS:
    assert set(labels[target].unique()).issubset({0, 1}), (
        f"Unexpected labels in {target}"
    )

# One study vector = mean of its nine slice embeddings.
study_features = features.mean(axis=1)
feature_df = pd.DataFrame(study_features)
feature_df.insert(0, ID_COL, study_ids)

# Join explicitly by study ID; never rely on row order.
data = (
    labels.copy()
    .assign(**{ID_COL: labels[ID_COL].astype(str)})
    .merge(
        feature_df.assign(**{ID_COL: feature_df[ID_COL].astype(str)}),
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        folds[[ID_COL, "fold"]].assign(
            **{ID_COL: folds[ID_COL].astype(str)}
        ),
        on=ID_COL,
        validate="one_to_one"
    )
)

feature_cols = list(range(768))
assert len(data) == 58 and data[ID_COL].is_unique
assert data[feature_cols].shape == (58, 768)

print("Input checks passed.")
print("Studies:", len(data))
print("Fold sizes:", data.groupby("fold").size().to_dict())
print("Features per study:", len(feature_cols))
print("Targets:", len(TARGETS))

# ---------------- Five-fold out-of-fold predictions ----------------
oof_parts = []
fold_metric_rows = []

for fold_id in range(N_FOLDS):
    train = data[data["fold"] != fold_id].copy()
    valid = data[data["fold"] == fold_id].copy()

    assert set(train[ID_COL]).isdisjoint(set(valid[ID_COL]))
    assert len(train) + len(valid) == 58

    print(
        f"\nFold {fold_id}: "
        f"train={len(train)}, validation={len(valid)}"
    )

    for target in TARGETS:
        y_train = train[target].astype(int).to_numpy()
        y_valid = valid[target].astype(int).to_numpy()

        # AUC requires both classes in validation; training must also
        # contain both classes for logistic regression.
        if len(np.unique(y_train)) != 2:
            raise ValueError(
                f"Fold {fold_id}, {target}: training lacks both classes"
            )
        if len(np.unique(y_valid)) != 2:
            raise ValueError(
                f"Fold {fold_id}, {target}: validation lacks both classes"
            )

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=2000,
                random_state=SEED
            )
        )

        model.fit(train[feature_cols], y_train)
        probabilities = model.predict_proba(valid[feature_cols])[:, 1]

        auc = roc_auc_score(y_valid, probabilities)

        oof_parts.append(pd.DataFrame({
            ID_COL: valid[ID_COL].to_numpy(),
            "fold": fold_id,
            "target": target,
            "y_true": y_valid,
            "y_prob": probabilities
        }))

        fold_metric_rows.append({
            "fold": fold_id,
            "target": target,
            "n_train": len(train),
            "n_validation": len(valid),
            "validation_positive": int(y_valid.sum()),
            "validation_negative": int((y_valid == 0).sum()),
            "auc": float(auc)
        })

oof = pd.concat(oof_parts, ignore_index=True)
fold_metrics = pd.DataFrame(fold_metric_rows)

# Each study-target pair must receive exactly one held-out prediction.
assert len(oof) == 58 * len(TARGETS)
assert not oof.duplicated([ID_COL, "target"]).any()
assert np.isfinite(oof["y_prob"]).all()
assert oof["y_prob"].between(0, 1).all()

# ---------------- Aggregate metrics ----------------
per_target_rows = []

for target in TARGETS:
    part = oof[oof["target"] == target]
    auc = roc_auc_score(part["y_true"], part["y_prob"])

    per_target_rows.append({
        "target": target,
        "n_studies": len(part),
        "positive": int(part["y_true"].sum()),
        "negative": int((part["y_true"] == 0).sum()),
        "oof_auc": float(auc)
    })

per_target = pd.DataFrame(per_target_rows)
overall_oof_macro_auc = float(per_target["oof_auc"].mean())

per_fold_macro = (
    fold_metrics.groupby("fold")["auc"]
    .mean()
    .rename("macro_auc")
    .reset_index()
)

summary = pd.DataFrame([{
    "experiment": "EXP-011",
    "method": "DINOv2 mean-slice features + logistic regression",
    "n_studies": len(data),
    "n_folds": N_FOLDS,
    "n_targets": len(TARGETS),
    "overall_oof_macro_auc": overall_oof_macro_auc,
    "mean_fold_macro_auc": float(per_fold_macro["macro_auc"].mean()),
    "std_fold_macro_auc": float(per_fold_macro["macro_auc"].std(ddof=1))
}])

# ---------------- Save auditable outputs ----------------
oof.to_csv(OUT / "exp011_oof_predictions.csv", index=False)
fold_metrics.to_csv(OUT / "exp011_fold_metrics.csv", index=False)
per_target.to_csv(OUT / "exp011_per_target_auc.csv", index=False)
per_fold_macro.to_csv(OUT / "exp011_per_fold_macro_auc.csv", index=False)
summary.to_csv(OUT / "exp011_summary.csv", index=False)

config = {
    "experiment": "EXP-011",
    "fold_source": str(FOLD_PATH),
    "fold_assignments_regenerated": False,
    "feature_source": str(CACHE / "features.npy"),
    "feature_aggregation": "mean across 9 slice embeddings",
    "classifier": "StandardScaler + LogisticRegression",
    "C": 1.0,
    "class_weight": "balanced",
    "max_iter": 2000,
    "random_state": SEED,
    "n_splits": N_FOLDS,
    "n_studies": len(data),
    "targets": TARGETS,
    "oof_predictions_per_study_target": True
}
(OUT / "exp011_config.json").write_text(
    json.dumps(config, indent=2)
)

print("\n=== PER-TARGET OOF AUC ===")
print(per_target.to_string(index=False))

print("\n=== PER-FOLD MACRO-AUC ===")
print(per_fold_macro.to_string(index=False))

print("\n=== EXP-011 SUMMARY ===")
print(summary.to_string(index=False))

print("\nSaved results to:", OUT)
print("Files:", sorted(p.name for p in OUT.iterdir()))


Input checks passed.
Studies: 58
Fold sizes: {0: 13, 1: 10, 2: 12, 3: 12, 4: 11}
Features per study: 768
Targets: 12

Fold 0: train=45, validation=13

Fold 1: train=48, validation=10

Fold 2: train=46, validation=12

Fold 3: train=46, validation=12

Fold 4: train=47, validation=11

=== PER-TARGET OOF AUC ===
          target  n_studies  positive  negative  oof_auc
             ACL         58        24        34 0.609069
             MCL         58         9        49 0.421769
 Medial Meniscus         58        26        32 0.474760
Lateral Meniscus         58        23        35 0.506832
       Medial OA         58        15        43 0.755039
      Lateral OA         58        11        47 0.740812
           PF OA         58        21        37 0.543115
        Effusion         58        35        23 0.642236
       Synovitis         58        27        31 0.467145
         Baker's         58        12        46 0.695652
       Contusion         58        19        39 0.539811
      


### EXP-011 results and interpretation

**Status:** Completed successfully.

- Studies: 58
- Targets: 12
- Folds: 5, using the original audited EXP-010 assignments
- Features: mean of nine frozen DINOv2 slice embeddings
- Classifier: StandardScaler + balanced logistic regression
- Overall pooled out-of-fold macro-AUC: **0.5893**
- Mean fold macro-AUC: **0.5941**
- Standard deviation of fold macro-AUC: **0.0548**

**Interpretation:** Performance varies considerably by target. Medial OA (0.7550) and Lateral OA (0.7408) have the highest OOF AUCs, while MCL (0.4218), Synovitis (0.4671), and Medial Meniscus (0.4748) are below 0.5.

The sample contains only 58 fully labelled studies, so these estimates are uncertain. The pooled OOF AUC and mean fold AUC are different aggregation methods and should both be retained.

**Decision:** Keep EXP-011 as the first five-fold image-only reference. Do not change the folds or tune hyperparameters based on these results alone.


## EXP-011 (continued) — TF-IDF report-text baseline on fixed OOF folds

Evaluate report text using the same five EXP-010 study-level folds as the DINOv2 image-only baseline. Fit the TF-IDF vocabulary separately within each training fold, then train one balanced logistic-regression classifier per target. Save out-of-fold predictions, per-target AUC, per-fold macro-AUC, configuration, and summary. This is a diagnostic experiment only: test-time reports are unavailable.

In [1]:

# EXP-011 (continued): TF-IDF report-text baseline on fixed OOF folds.
# GPU OFF. Permanent experiment cell. Do not delete after running.

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ---------------- Configuration ----------------
DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
FOLD_PATH = Path("/kaggle/input/datasets/maratiusupov/exp010-folds/exp010_folds.csv")
OUT = Path("/kaggle/working/exp011_text_cv_results")
OUT.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

SEED = 42
N_FOLDS = 5

# ---------------- Load and validate inputs ----------------
train = pd.read_csv(DATA / "train.csv")
folds = pd.read_csv(FOLD_PATH)

required = ["StudyInstanceUID", "Report", *TARGETS]
missing = [c for c in required if c not in train.columns]
assert not missing, f"Missing train.csv columns: {missing}"
assert {"StudyInstanceUID", "fold"}.issubset(folds.columns), (
    f"Unexpected fold columns: {folds.columns.tolist()}"
)
assert train["StudyInstanceUID"].is_unique, "Duplicate study IDs in train.csv"
assert folds["StudyInstanceUID"].is_unique, "Duplicate study IDs in fold file"

# Use only studies with all official labels observed.
gold = train.dropna(subset=TARGETS).copy()
gold["StudyInstanceUID"] = gold["StudyInstanceUID"].astype(str)
folds["StudyInstanceUID"] = folds["StudyInstanceUID"].astype(str)

assert len(gold) == 58, f"Expected 58 fully labelled studies; got {len(gold)}"
assert set(gold["StudyInstanceUID"]) == set(folds["StudyInstanceUID"]), (
    "Fold IDs do not exactly match the complete-label studies"
)
assert set(folds["fold"].unique()) == set(range(N_FOLDS)), (
    f"Unexpected fold IDs: {sorted(folds['fold'].unique())}"
)

data = gold.merge(
    folds[["StudyInstanceUID", "fold"]],
    on="StudyInstanceUID",
    how="inner",
    validate="one_to_one"
).sort_values("StudyInstanceUID").reset_index(drop=True)

assert len(data) == 58
assert data["Report"].notna().all(), "Missing reports among gold-labelled studies"
assert data["Report"].astype(str).str.strip().ne("").all(), (
    "Empty reports among gold-labelled studies"
)
assert data[TARGETS].isin([0, 1]).all().all(), "Labels must be binary"

print("Input checks passed.")
print("Gold-labelled studies:", len(data))
print("Fold sizes:", data["fold"].value_counts().sort_index().to_dict())
print("Report-text input: training-study reports only")
print("Test reports loaded: NO")

# ---------------- Fixed-fold OOF evaluation ----------------
oof_rows = []
fold_target_rows = []
fold_macro_rows = []

for fold in range(N_FOLDS):
    train_part = data.loc[data["fold"] != fold].copy()
    val_part = data.loc[data["fold"] == fold].copy()

    assert set(train_part["StudyInstanceUID"]).isdisjoint(
        set(val_part["StudyInstanceUID"])
    )

    # Fit vocabulary only on this fold's training reports.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        max_features=20000,
        sublinear_tf=True
    )
    X_train = vectorizer.fit_transform(train_part["Report"].astype(str))
    X_val = vectorizer.transform(val_part["Report"].astype(str))

    fold_aucs = []
    print(
        f"Fold {fold}: train={len(train_part)}, "
        f"validation={len(val_part)}, vocabulary={len(vectorizer.vocabulary_)}"
    )

    for target in TARGETS:
        y_train = train_part[target].to_numpy(dtype=int)
        y_val = val_part[target].to_numpy(dtype=int)

        assert len(np.unique(y_train)) == 2, (
            f"{target}: training fold {fold} lacks one class"
        )
        assert len(np.unique(y_val)) == 2, (
            f"{target}: validation fold {fold} lacks one class"
        )

        model = LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED
        )
        model.fit(X_train, y_train)
        probabilities = model.predict_proba(X_val)[:, 1]
        auc = float(roc_auc_score(y_val, probabilities))
        fold_aucs.append(auc)

        fold_target_rows.append({
            "fold": fold,
            "target": target,
            "n_train": len(train_part),
            "n_validation": len(val_part),
            "validation_positive": int(y_val.sum()),
            "validation_negative": int((y_val == 0).sum()),
            "auc": auc
        })

        for study_id, y_true, y_prob in zip(
            val_part["StudyInstanceUID"], y_val, probabilities
        ):
            oof_rows.append({
                "StudyInstanceUID": study_id,
                "fold": fold,
                "target": target,
                "y_true": int(y_true),
                "y_prob": float(y_prob)
            })

    fold_macro = float(np.mean(fold_aucs))
    fold_macro_rows.append({"fold": fold, "macro_auc": fold_macro})
    print(f"  Fold macro-AUC: {fold_macro:.6f}")

# ---------------- Summaries and integrity checks ----------------
oof = pd.DataFrame(oof_rows)
fold_metrics = pd.DataFrame(fold_target_rows)
per_fold = pd.DataFrame(fold_macro_rows)

assert len(oof) == 58 * len(TARGETS), f"Unexpected OOF row count: {len(oof)}"
assert not oof.duplicated(["StudyInstanceUID", "target"]).any()
assert oof.groupby("target")["StudyInstanceUID"].nunique().eq(58).all()

per_target_rows = []
for target in TARGETS:
    part = oof.loc[oof["target"] == target]
    auc = float(roc_auc_score(part["y_true"], part["y_prob"]))
    per_target_rows.append({
        "target": target,
        "n_studies": int(part["StudyInstanceUID"].nunique()),
        "positive": int(part["y_true"].sum()),
        "negative": int((part["y_true"] == 0).sum()),
        "oof_auc": auc
    })

per_target = pd.DataFrame(per_target_rows)
pooled_oof_macro_auc = float(per_target["oof_auc"].mean())
mean_fold_macro_auc = float(per_fold["macro_auc"].mean())
std_fold_macro_auc = float(per_fold["macro_auc"].std(ddof=0))

summary = pd.DataFrame([{
    "experiment": "EXP-011",
    "method": "TF-IDF report text + logistic regression",
    "n_studies": 58,
    "n_folds": N_FOLDS,
    "n_targets": len(TARGETS),
    "pooled_oof_macro_auc": pooled_oof_macro_auc,
    "mean_fold_macro_auc": mean_fold_macro_auc,
    "std_fold_macro_auc": std_fold_macro_auc
}])

config = {
    "experiment": "EXP-011",
    "method": "TF-IDF report text + logistic regression",
    "fold_source": str(FOLD_PATH),
    "fold_assignments_regenerated": False,
    "n_studies": 58,
    "n_folds": N_FOLDS,
    "targets": TARGETS,
    "text_preprocessing": {
        "lowercase": True,
        "strip_accents": "unicode",
        "ngram_range": [1, 2],
        "min_df": 1,
        "max_features": 20000,
        "sublinear_tf": True,
        "fit_vocabulary_within_each_training_fold": True
    },
    "classifier": "LogisticRegression",
    "C": 1.0,
    "class_weight": "balanced",
    "max_iter": 2000,
    "random_state": SEED,
    "test_reports_used": False,
    "purpose": "diagnostic text-only baseline; not usable as test-time input"
}

oof.to_csv(OUT / "exp011_text_oof_predictions.csv", index=False)
fold_metrics.to_csv(OUT / "exp011_text_fold_metrics.csv", index=False)
per_fold.to_csv(OUT / "exp011_text_per_fold_macro_auc.csv", index=False)
per_target.to_csv(OUT / "exp011_text_per_target_auc.csv", index=False)
summary.to_csv(OUT / "exp011_text_summary.csv", index=False)
(OUT / "exp011_text_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)

print("\n=== PER-TARGET POOLED OOF AUC ===")
print(per_target.to_string(index=False))
print("\n=== PER-FOLD MACRO AUC ===")
print(per_fold.to_string(index=False))
print("\n=== EXP-011 TEXT SUMMARY ===")
print(summary.to_string(index=False))
print("\nSaved six result files to:", OUT)
print("Compare pooled OOF macro-AUC with image-only EXP-011: 0.589270")
print("This is a diagnostic comparison; report text is unavailable at test time.")


Input checks passed.
Gold-labelled studies: 58
Fold sizes: {0: 13, 1: 10, 2: 12, 3: 12, 4: 11}
Report-text input: training-study reports only
Test reports loaded: NO
Fold 0: train=45, validation=13, vocabulary=5852
  Fold macro-AUC: 0.582970
Fold 1: train=48, validation=10, vocabulary=5911
  Fold macro-AUC: 0.546806
Fold 2: train=46, validation=12, vocabulary=5898
  Fold macro-AUC: 0.547492
Fold 3: train=46, validation=12, vocabulary=5660
  Fold macro-AUC: 0.604384
Fold 4: train=47, validation=11, vocabulary=6071
  Fold macro-AUC: 0.600099

=== PER-TARGET POOLED OOF AUC ===
          target  n_studies  positive  negative  oof_auc
             ACL         58        24        34 0.681373
             MCL         58         9        49 0.371882
 Medial Meniscus         58        26        32 0.671875
Lateral Meniscus         58        23        35 0.509317
       Medial OA         58        15        43 0.719380
      Lateral OA         58        11        47 0.599613
           PF OA    